In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from sklearn.mixture import GaussianMixture

import umap
import hdbscan

In [ ]:
df = pd.read_parquet("../data/features/features_windowed_final.parquet")

print("Dataset shape:", df.shape)

df.head()
df.info()
df.describe().T.head(10)

In [ ]:
meta_cols = [
    "trip_id",
    "start_time"
]

In [ ]:
features = df.drop(columns=meta_cols)

print("Feature count:", features.shape[1])

In [ ]:
missing = features.isna().sum().sort_values(ascending=False)

print(missing.head(10))

In [ ]:
for col in features.columns:

    q1 = features[col].quantile(0.01)
    q99 = features[col].quantile(0.99)

    features[col] = features[col].clip(q1, q99)

In [ ]:
cluster_features = [

    # Vehicle dynamics
    "speed_mean",
    "speed_std",
    "speed_stability",
    "acceleration_mean",
    "acceleration_std",
    "acc_variability",
    "jerk_intensity",

    # Engine / load
    "rpm_mean",
    "rpm_std",
    "engine_load",
    "map_mean",
    "load_indicator",
    "rpm_speed_ratio_norm",

    # Driver control
    "throttle_mean",
    "throttle_smoothness",
    "throttle_speed_ratio",

    # Driving events
    "harsh_acc_events",
    "harsh_brake_events",

    # Behavioral indices
    "aggression_score",
    "control_instability"

]

In [ ]:
X = df[cluster_features]

print("Clustering features:", len(cluster_features))

In [ ]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
import umap

umap_model = umap.UMAP(

    n_neighbors=50,
    min_dist=0.05,
    n_components=10,
    metric="euclidean",
    random_state=42

)

X_umap = umap_model.fit_transform(X_scaled)

print("UMAP shape:", X_umap.shape)

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=5,
    n_init=50,
    random_state=42
)

clusters = kmeans.fit_predict(X_umap)

df["context_cluster"] = clusters

In [ ]:
df["context_cluster"].value_counts()

In [ ]:
from sklearn.metrics import silhouette_score

score = silhouette_score(
    X_umap[df.index],
    df["context_cluster"]
)

print("Silhouette Score:", score)

In [ ]:
umap_vis = umap.UMAP(

    n_neighbors=50,
    min_dist=0.05,
    n_components=2,
    random_state=42

)

X_vis = umap_vis.fit_transform(X_scaled)

In [ ]:
plt.figure(figsize=(8,6))

sns.scatterplot(

    x=X_vis[:,0],
    y=X_vis[:,1],
    hue=df["context_cluster"],
    palette="tab10",
    s=10

)

plt.title("Driving Context Clusters")

plt.show()

In [ ]:
cluster_summary = df.groupby("context_cluster")[cluster_features].mean()

cluster_summary

In [ ]:
context_labels = {
    0: "Stop-Go Traffic",
    1: "Normal City Driving",
    2: "Aggressive Driving",
    3: "Highway Cruising",
    4: "Heavy Engine Load"
}

df["context_label"] = df["context_cluster"].map(context_labels)

In [ ]:
df["context_label"].value_counts()

In [ ]:
df["context_label"].value_counts().plot(
    kind="bar",
    figsize=(8,5)
)

plt.title("Driving Context Distribution")
plt.xlabel("Driving Context")
plt.ylabel("Number of Windows")

plt.show()

In [ ]:
df["context_id"] = df["context_cluster"]

df.to_parquet(
    "../data/features/final_features_with_context.parquet",
    index=False
)

In [ ]:
df.to_csv(
    "../data/features/final_features_with_context.csv",
    index=False
)